# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading and analysis of the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed. Uncomment below if running for the first time.
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL of the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Number of record sets: {len(metadata.record_sets)}")

## 2. Data Overview
Let’s examine the available record sets, fields, and columns within the dataset, referring to their `@id` values for unambiguous access.

In [ ]:
# List all record set @id and field @id inside each
for record_set in metadata.record_sets:
    print(f"Record Set: {record_set['@id']} (name: {record_set.get('name', '')})")
    fields = record_set.get('fields', [])
    if not fields:
        print("  No fields in this record set.")
    for field in fields:
        print(f"  Field: {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
    columns = record_set.get('columns', [])
    for column in columns:
        print(f"  Column: {column['@id']} (name: {column.get('name', '')}, dataType: {column.get('dataType', '')})")
    print('')

# Choose the first record set @id for demonstration (if any)
first_record_set_id = None
if len(metadata.record_sets) > 0:
    first_record_set_id = metadata.record_sets[0]['@id']
    print('Example record set @id:', first_record_set_id)

# Show an example record from the first record set (if present)
if first_record_set_id:
    print('Example record:')
    for rec in dataset.records(record_set=first_record_set_id):
        print(rec)
        break

## 3. Data Extraction
We will load all rows from each record set, placing the resulting data into a pandas DataFrame.

**Note:** All references use the `@id` field for record sets, as shown in the overview.

In [ ]:
# Gather list of all record set @id
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

# Load data from each record set into a DataFrame, indexed by @id
dfs = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if len(records) > 0:
        dfs[rsid] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set @id: {rsid}")
        print(f"Columns: {dfs[rsid].columns.tolist()}")
        print(dfs[rsid].head(2), '\n')

if first_record_set_id and first_record_set_id in dfs:
    print('First record set fields:', dfs[first_record_set_id].columns.tolist())
    display(dfs[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
This section demonstrates data filtering, normalization, and grouping operations using one of the numeric fields by referencing its field `@id`.

**Note:** Please adjust `numeric_field_id` and `group_field_id` to match actual columns/fields in your specific record set.

In [ ]:
# ---- Please adapt these values based on your dataset's field @id ---- #
# For illustration, use likely field ids. You may change as per above overview output.
# Let's fetch the first available numeric field
import numpy as np

numeric_field_id = None
group_field_id = None

if first_record_set_id and first_record_set_id in dfs:
    df = dfs[first_record_set_id]
    # Try to auto-detect a numeric field
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            continue
    # Try to detect a categorical field for grouping
    for col in df.columns:
        if col != numeric_field_id and (df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col])):
            group_field_id = col
            break
    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")

    # Set a threshold for filtering
    if numeric_field_id is not None:
        try:
            threshold = np.percentile(df[numeric_field_id].dropna(), 75) # 75th percentile
        except Exception:
            threshold = 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group-by if group field exists
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} by {group_field_id}:")
            print(grouped.head())

## 5. Visualization
Let’s plot the distribution of the selected numeric field and a grouped bar plot if a group field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if first_record_set_id and first_record_set_id in dfs and numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(dfs[first_record_set_id][numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Grouped bar plot if both fields exist
if first_record_set_id and first_record_set_id in dfs and numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 4))
    group_avg = dfs[first_record_set_id].groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    sns.barplot(x=group_avg.index, y=group_avg.values)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook illustrated how to use the `mlcroissant` library to explore and process a dataset described by a Croissant schema by referencing all relevant entities through their `@id` fields. You can further refine the exploration by referencing specific record sets, fields, and analysis goals as required for your research.

**Key takeaways:**
- The dataset exposes detailed clinicopathological and molecular information about second primary colorectal cancer, accessible via Croissant schema entities.
- All data operations and column references should use the canonical `@id` for full reproducibility.
- Further steps could include building predictive ML models or detailed statistical analyses using this preprocessed data.